# ROE-TAPE Research Objective Notebook
## Routed Objective Experts for TAPE

### Working Title
**ROE-TAPE: A Staged Multi-Objective Reinforcement Learning Framework for Portfolio Optimization with Regime-Routed Objective Experts, Separate Critics, and Simplex-Stable Dirichlet Policies**

This notebook is a research-design document grounded in the current codebase state. It is not a generic idea dump. It is intended to define a publishable research objective, the core novelty claims, the exact methodology, and the ablation program required to make the paper defensible.


## 1. Scope
This notebook captures four things:

1. The **publishable research objective** that best matches the current codebase and conversation history.
2. The **core novel ideas** that should become the paper's central contribution.
3. The **methodology and evaluation protocol** needed to test those ideas rigorously.
4. The **ablation suite** needed to separate real contributions from incidental gains.

The design principle is simple:
- do not publish a loose collection of heuristics
- publish a coherent multi-objective RL framework with explicit architectural and methodological contributions


In [ ]:
from pprint import pprint
from src.config import build_run19_config, assert_run19_config

cfg = build_run19_config('phase1')
assert_run19_config(cfg)

ap = cfg['agent_params']
ppo = ap['ppo_params']
tp = cfg['training_params']
env = cfg['environment_params']

snapshot = {
    'run_lineage': 'run19',
    'assets': cfg['ASSET_TICKERS'],
    'num_assets': cfg['NUM_ASSETS'],
    'architecture': ap['actor_critic_type'],
    'objective_experts_enabled': ap.get('objective_experts_enabled'),
    'objective_expert_names': ap.get('objective_expert_names'),
    'dirichlet_alpha_activation': ap.get('dirichlet_alpha_activation'),
    'dirichlet_softplus_alpha_floor': ap.get('dirichlet_softplus_alpha_floor'),
    'dirichlet_softplus_alpha_scale': ap.get('dirichlet_softplus_alpha_scale'),
    'reward_component_schedule': tp.get('reward_component_schedule'),
    'target_turnover': env.get('target_turnover'),
}

pprint(snapshot)


## 2. Current Codebase State
The current codebase has already moved beyond a monolithic PPO portfolio policy.

### What now exists
- Shared `TCN_FUSION` encoder.
- Separate **actor experts**:
  - return
  - risk
  - discipline
- Separate **critic heads** aligned to those objectives.
- A **regime router** that blends expert logits into the final policy.
- A staged **TAPE reward curriculum**.
- A strengthened **cross-softplus Dirichlet alpha generator**.
- Covariance summaries plus **per-asset PC loading features**.
- Regime-stratified deterministic evaluation and regime-stratified stochastic confirmation infrastructure.
- Dirichlet numerical hardening and expanded Run19 logging.

### What does not yet define the publication contribution by itself
- A code implementation is not a research objective.
- A staged reward stack is not enough on its own.
- The paper needs a clear causal claim about *why* this architecture is better and *which parts* are responsible.


## 3. Recommended Publishable Research Objective
### Main Research Objective
Develop and validate a **staged, regime-routed, multi-objective reinforcement learning framework** for portfolio optimization in which:

- distinct portfolio objectives are **architecturally separated**
- specialized actor and critic heads learn those objectives with cleaner gradients
- a learned router blends expert behavior according to market state
- the final policy remains simplex-valid through a stabilized Dirichlet parameterization

### Core Research Question
**Can multi-objective portfolio RL be made more stable, interpretable, and regime-robust by replacing a monolithic actor-critic with objective-specific experts, separate critics, and regime-conditioned routing under a staged TAPE curriculum?**

### Why this is publishable
This objective is stronger than "we improved Sharpe with a new RL model" because it makes a structural claim:

> portfolio objectives such as return seeking, risk control, and trading discipline should not be forced through a single policy/value pathway.

That is a concrete architectural thesis, not a metric-chasing recipe.


## 4. Proposed Paper-Level Contributions
### Contribution 1: TAPE as a Structured Multi-Objective RL Framework
The paper should explicitly reframe TAPE from:
- a scalar reward stack

to:
- a **structured objective system**

Mapped objectives:
- `c0`: base return
- `c1`: DSR / benchmark-aligned risk shaping
- `c2`: turnover / discipline shaping
- terminal TAPE: episode-level supervision on the blended policy

### Contribution 2: Routed Objective Experts
Instead of one actor head, use:
- return expert
- risk expert
- discipline expert

Instead of one critic, use:
- return critic
- risk critic
- discipline critic

The router blends the expert policies according to market state.

### Contribution 3: Shared Encoder, Separate Objective Adapters
The architecture is not three separate agents. It is:
- a shared temporal market encoder
- objective-specific adapters and heads

This is important because it positions the contribution as:
- efficient shared representation learning
- with targeted downstream objective separation

### Contribution 4: Regime-Routed Policy Blending
The router transforms regime and context summaries into soft expert weights.
This creates a continuous regime-dependent strategy rather than a hard-switch controller.

### Contribution 5: Simplex-Stable Dirichlet Policy Engineering
The final portfolio policy remains simplex-valid while using:
- `cross_softplus`
- cross-sectional standardization
- tuned concentration scaling
- numerical stabilization around Dirichlet log-prob evaluation

This is not just an implementation detail. It can be framed as a practical contribution to training simplex-constrained RL policies.


## 5. Proposed Name and Framing
### Recommended model name
**ROE-TAPE**

Expanded:
**Routed Objective Experts for TAPE**

### Recommended framing sentence
ROE-TAPE is a staged multi-objective portfolio RL architecture that combines:
- a shared temporal market encoder
- objective-specific actor and critic experts
- regime-conditioned routing
- simplex-stable Dirichlet policy outputs

### Why this name works
- it foregrounds the main novelty: routed objective experts
- it preserves continuity with TAPE
- it is compact enough for a title, figures, and ablation tables


## 6. Formal Methodology

### 6.1 State Representation

The observation at time $t$ is a structured tensor composed of per-asset and global signals.

**Per-asset feature matrix.** For $N$ assets and $F$ per-asset features over a lookback window of $T$ timesteps:

$$\mathbf{X}^{\text{asset}}_t \in \mathbb{R}^{N \times T \times F}$$

Each asset $i$ receives a feature vector $\mathbf{x}^{(i)}_t \in \mathbb{R}^F$ comprising:

| Family | Features | Count |
|--------|----------|-------|
| Log returns | $r^{(i)}_{\tau}$ for $\tau \in \{1,5,10,21\}$ | 4 |
| Rolling statistics | $\mu_{\tau}, \sigma_{\tau}, z_{\tau}, \text{skew}_{\tau}$ | 4 |
| Technical indicators | RSI, MACD signal, BB width, ATR, OBV $\Delta$, VWAP ratio | 6 |
| Candlestick patterns | Body ratio, upper/lower shadow, gap, intraday range, volume ratio, close position | 7 |
| Regime signals | Volatility regime, trend strength, drawdown depth, recovery ratio, HMM state, etc. | 8 |
| Cross-sectional | Relative strength rank $\text{rank}(r^{(i)}_{21}) / N$ | 1 |
| Alpha signals | Momentum scores, mean-reversion z-scores, quality factors | 5 |
| Quant alpha | Factor exposures, residual alpha estimates | 6 |
| Covariance loadings | Per-asset PC1/PC2 loadings from rolling covariance | 2 |

Total per-asset features: $F \approx 55$.

**Global context vector.** Market-wide signals shared across all assets:

$$\mathbf{x}^{\text{global}}_t \in \mathbb{R}^{G}$$

comprising covariance eigenvalues $\lambda_1, \lambda_2$, effective rank, mean pairwise correlation, and macro regime indicators. Typically $G \approx 4$.

**Flat observation.** The environment emits a flat vector:

$$\mathbf{o}_t = [\text{vec}(\mathbf{X}^{\text{asset}}_t) \,\|\, \mathbf{x}^{\text{global}}_t] \in \mathbb{R}^{N \cdot F + G}$$

For $N=10, F=55, G=4$: $\dim(\mathbf{o}_t) = 554$.

**Normalization.** Each feature is normalized by family-specific routing:
- **Bounded features** (e.g., RSI): min-max scaling to $[0, 1]$
- **Unbounded features** (e.g., log returns): robust winsorization at the 1st/99th percentile followed by standard scaling:

$$\hat{x} = \frac{\text{clip}(x, q_{0.01}, q_{0.99}) - \mu}{\sigma + \epsilon}$$

---

### 6.2 Shared Encoder: TCN-FUSION

The shared encoder processes the structured observation into a latent representation $\mathbf{H} \in \mathbb{R}^{N \times d}$ where $d$ is the fusion embedding dimension.

#### 6.2.1 Per-Asset TCN Stack (Shared Weights)

The per-asset feature sequences are reshaped and processed through a weight-shared TCN:

$$\mathbf{X}^{(i)} \in \mathbb{R}^{T \times F}, \quad i = 1, \ldots, N$$

Each TCN block $l$ applies causal dilated convolutions with residual connections:

$$\mathbf{h}^{(i,l)} = \text{TCNBlock}_l(\mathbf{h}^{(i,l-1)}), \quad \mathbf{h}^{(i,0)} = \mathbf{X}^{(i)}$$

A single TCN block consists of:

$$\begin{aligned}
\mathbf{z} &= \text{Conv1D}(\mathbf{h}; \text{filters}=d_{\text{tcn}}, \text{kernel}=k, \text{dilation}=d_l, \text{causal padding}) \\
\mathbf{z} &= \text{LayerNorm}(\mathbf{z}) \\
\mathbf{z} &= \text{GELU}(\mathbf{z}) \\
\mathbf{z} &= \text{Dropout}(\mathbf{z}, p) \\
\mathbf{h}^{(i,l)} &= \mathbf{h}^{(i,l-1)} + \mathbf{z} \quad \text{(residual connection)}
\end{aligned}$$

Dilation rates grow exponentially: $d_l = 2^l$ for $l = 0, 1, \ldots, L-1$.

The receptive field of the TCN stack is:

$$\text{RF} = 1 + \sum_{l=0}^{L-1} (k - 1) \cdot 2^l$$

After the TCN stack, temporal pooling aggregates across the time axis:

$$\mathbf{e}^{(i)} = \text{Pool}_T(\mathbf{h}^{(i,L)}) \in \mathbb{R}^{d_{\text{tcn}}}$$

followed by a linear projection:

$$\mathbf{e}^{(i)} = \mathbf{W}_{\text{proj}} \mathbf{e}^{(i)} + \mathbf{b}_{\text{proj}} \in \mathbb{R}^d$$

The per-asset embeddings are stacked:

$$\mathbf{E} = [\mathbf{e}^{(1)}; \mathbf{e}^{(2)}; \ldots; \mathbf{e}^{(N)}] \in \mathbb{R}^{N \times d}$$

**Asset identity embeddings.** Optionally, a learnable identity embedding is added:

$$\mathbf{E} \leftarrow \mathbf{E} + \mathbf{I}_{\text{asset}} \in \mathbb{R}^{N \times d}$$

where $\mathbf{I}_{\text{asset}}$ is a trainable parameter matrix.

#### 6.2.2 Cross-Asset Mixer (Self-Attention)

The cross-asset mixer applies Transformer-style self-attention across the $N$ asset tokens with pre-norm residual blocks:

$$\begin{aligned}
\tilde{\mathbf{E}} &= \text{LayerNorm}(\mathbf{E}) \\
\mathbf{Q} = \tilde{\mathbf{E}}\mathbf{W}_Q, \quad \mathbf{K} &= \tilde{\mathbf{E}}\mathbf{W}_K, \quad \mathbf{V} = \tilde{\mathbf{E}}\mathbf{W}_V \\
\text{Attn}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) &= \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V} \\
\mathbf{E} &\leftarrow \mathbf{E} + \text{Dropout}(\text{MHA}(\tilde{\mathbf{E}}, \tilde{\mathbf{E}}))
\end{aligned}$$

where MHA denotes multi-head attention with $H$ heads and key dimension $d_k = d / H$.

This is followed by a feed-forward residual block:

$$\begin{aligned}
\hat{\mathbf{E}} &= \text{LayerNorm}(\mathbf{E}) \\
\mathbf{F} &= \text{GELU}(\hat{\mathbf{E}}\mathbf{W}_1 + \mathbf{b}_1) \\
\mathbf{F} &= \text{Dropout}(\mathbf{F}\mathbf{W}_2 + \mathbf{b}_2) \\
\mathbf{E} &\leftarrow \mathbf{E} + \mathbf{F}
\end{aligned}$$

where the hidden dimension is $d_{\text{ff}} = \lfloor d \cdot \text{expansion} \rfloor$.

#### 6.2.3 Global Context Branch

The global context sequence is processed through an optional memory layer, then pooled and projected:

$$\mathbf{c} = \text{Dropout}(\mathbf{W}_g \cdot \text{Pool}_T(\mathbf{x}^{\text{global}}) + \mathbf{b}_g) \in \mathbb{R}^{d}$$

**Context cross-attention.** When enabled, the global context is injected into the asset tokens via cross-attention:

$$\begin{aligned}
\mathbf{c}_{\text{token}} &= \mathbf{W}_c \mathbf{c} + \mathbf{b}_c \in \mathbb{R}^{1 \times d} \\
\mathbf{E} &\leftarrow \mathbf{E} + \text{MHA}(\text{query}=\mathbf{E},\; \text{context}=\mathbf{c}_{\text{token}})
\end{aligned}$$

**Legacy gate fusion.** When cross-attention is disabled, a sigmoid gate blends asset and context:

$$\begin{aligned}
\bar{\mathbf{e}} &= \text{Pool}_N(\mathbf{E}) \in \mathbb{R}^d \\
g &= \sigma(\mathbf{W}_{\text{gate}}[\bar{\mathbf{e}} \,\|\, \mathbf{c}] + b_{\text{gate}}) \\
\mathbf{f} &= g \cdot \bar{\mathbf{e}} + (1 - g) \cdot \mathbf{c}
\end{aligned}$$

#### 6.2.4 FiLM Conditioning (Feature-wise Linear Modulation)

FiLM applies regime-dependent affine modulation to the shared features. Given a conditioning signal $\mathbf{z}$ (regime embedding or relative-strength signal):

$$\begin{aligned}
\mathbf{h} &= \text{GELU}(\text{GELU}(\text{LayerNorm}(\mathbf{z})\mathbf{W}_1)\mathbf{W}_2) \\
\boldsymbol{\gamma}_{\Delta} &= \gamma_{\max} \cdot \tanh(\mathbf{W}_{\gamma}\mathbf{h} + \mathbf{b}_{\gamma}) \\
\boldsymbol{\beta} &= \beta_{\max} \cdot \tanh(\mathbf{W}_{\beta}\mathbf{h} + \mathbf{b}_{\beta}) \\
\boldsymbol{\gamma} &= 1 + \boldsymbol{\gamma}_{\Delta}
\end{aligned}$$

The modulated output is:

$$\mathbf{x}_{\text{out}} = \mathbf{x} + \text{Dropout}((\boldsymbol{\gamma} - 1) \odot \mathbf{x} + \boldsymbol{\beta})$$

which simplifies to:

$$\mathbf{x}_{\text{out}} = \mathbf{x} + \text{Dropout}(\boldsymbol{\gamma}_{\Delta} \odot \mathbf{x} + \boldsymbol{\beta})$$

The clamped $\tanh$ output ensures $\boldsymbol{\gamma} \in [1 - \gamma_{\max},\; 1 + \gamma_{\max}]$ and $\boldsymbol{\beta} \in [-\beta_{\max},\; \beta_{\max}]$. Typical values: $\gamma_{\max} = 0.15$, $\beta_{\max} = 0.10$. Both $\mathbf{W}_{\gamma}$ and $\mathbf{W}_{\beta}$ are zero-initialized so that FiLM is initially identity.

**Market-relative FiLM.** For per-asset conditioning, the conditioning signal is the deviation from the cross-asset mean:

$$\begin{aligned}
\bar{\mathbf{e}} &= \frac{1}{N}\sum_{i=1}^N \mathbf{e}^{(i)} \\
\mathbf{z}^{(i)} &= \mathbf{e}^{(i)} - \bar{\mathbf{e}} + \mathbf{I}^{(i)}_{\text{asset}}
\end{aligned}$$

This enables each asset's representation to be modulated relative to the market average.

---

### 6.3 Objective-Specific Actor Heads (ROE)

From the shared encoder output $\mathbf{E} \in \mathbb{R}^{N \times d}$ and fused context $\mathbf{f} \in \mathbb{R}^d$, three expert paths produce per-objective logits.

#### 6.3.1 Expert Adapter

Each expert $k \in \{\text{return}, \text{risk}, \text{discipline}\}$ has a lightweight adapter applied to both asset-level and fused representations:

$$\begin{aligned}
\mathbf{A}_k &= \text{Dropout}(\text{GELU}(\text{LayerNorm}(\mathbf{E})\mathbf{W}^{\text{asset}}_k)) \in \mathbb{R}^{N \times d} \\
\mathbf{f}_k &= \text{Dropout}(\text{GELU}(\text{LayerNorm}(\mathbf{f})\mathbf{W}^{\text{fused}}_k)) \in \mathbb{R}^{d}
\end{aligned}$$

#### 6.3.2 Per-Asset Logit Heads

Each expert produces per-asset allocation logits via a per-asset head operating on its adapted asset features:

$$\begin{aligned}
\ell^{(i)}_k &= \mathbf{w}^{\text{logit}}_k \cdot \mathbf{a}^{(i)}_k + b^{\text{logit}}_k, \quad i = 1, \ldots, N \\
\ell^{\text{cash}}_k &= \mathbf{w}^{\text{cash}}_k \cdot \mathbf{f}_k + b^{\text{cash}}_k \\
\boldsymbol{\ell}_k &= [\ell^{(1)}_k, \ldots, \ell^{(N)}_k, \ell^{\text{cash}}_k] \in \mathbb{R}^{N+1}
\end{aligned}$$

The expert logit tensor is:

$$\mathbf{L} = [\boldsymbol{\ell}_{\text{ret}};\; \boldsymbol{\ell}_{\text{risk}};\; \boldsymbol{\ell}_{\text{disc}}] \in \mathbb{R}^{K \times (N+1)}$$

---

### 6.4 Objective-Specific Critic Heads

Each objective has a dedicated value function estimating the expected discounted return under its objective-specific reward stream:

$$V_k(s_t) = \mathbb{E}\left[\sum_{\tau=0}^{\infty} \gamma^\tau r^{(k)}_{t+\tau} \;\middle|\; s_t\right], \quad k \in \{\text{ret}, \text{risk}, \text{disc}\}$$

Each critic head is a separate MLP operating on the shared encoder output:

$$V_k(s_t) = \mathbf{w}_k^{(2)} \cdot \text{ReLU}(\mathbf{W}_k^{(1)} \mathbf{f} + \mathbf{b}_k^{(1)}) + b_k^{(2)}$$

The blended critic (used for the primary GAE computation) is:

$$V(s_t) = \sum_k g_k(s_t) \cdot V_k(s_t)$$

where $g_k$ are the router weights (Section 6.5).

---

### 6.5 Regime-Conditioned Router

The router produces soft expert gating weights conditioned on market state.

**Router input.** The router receives a concatenation of asset-pooled features, global context, and optional regime embedding:

$$\mathbf{r}_t = [\text{Pool}_N(\mathbf{E}) \,\|\, \mathbf{c} \,\|\, \mathbf{z}_{\text{regime}}] \in \mathbb{R}^{d_r}$$

**Router MLP.** A small MLP produces raw gating logits:

$$\begin{aligned}
\mathbf{h}_r &= \text{Dropout}(\text{GELU}(\mathbf{W}_r^{(1)} \mathbf{r}_t + \mathbf{b}_r^{(1)})) \\
\boldsymbol{\ell}_r &= \mathbf{W}_r^{(2)} \mathbf{h}_r + \mathbf{b}_r^{(2)} \in \mathbb{R}^K
\end{aligned}$$

**Expert masking.** A binary curriculum mask $\mathbf{m} \in \{0, 1\}^K$ disables inactive experts:

$$\tilde{\ell}_{r,k} = \begin{cases} \ell_{r,k} & \text{if } m_k = 1 \\ -\infty & \text{if } m_k = 0 \end{cases}$$

**Gating probabilities.** The final router probabilities are:

$$\mathbf{g} = \frac{\text{softmax}(\tilde{\boldsymbol{\ell}}_r) \odot \mathbf{m}}{\sum_k [\text{softmax}(\tilde{\boldsymbol{\ell}}_r)]_k \cdot m_k} \in \Delta^{K-1}$$

where $\Delta^{K-1}$ is the probability simplex.

**Policy logit blending.** The blended policy logits are:

$$\boldsymbol{\ell}_{\text{final}} = \sum_{k=1}^K g_k \cdot \boldsymbol{\ell}_k \in \mathbb{R}^{N+1}$$

---

### 6.6 Policy Output: Dirichlet Alpha Generation

The blended logits $\boldsymbol{\ell}_{\text{final}}$ are transformed into Dirichlet concentration parameters $\boldsymbol{\alpha} \in \mathbb{R}^{N+1}_{>0}$.

#### 6.6.1 Cross-Sectional Softplus (`cross_softplus`)

The primary activation in ROE-TAPE:

$$\begin{aligned}
\bar{\ell} &= \frac{1}{N+1}\sum_{j=1}^{N+1} \ell_j \quad \text{(cross-sectional mean)} \\
\hat{\ell}_j &= \ell_j - \bar{\ell} \quad \text{(centering)} \\
\end{aligned}$$

With optional cross-sectional standardization:

$$\hat{\ell}_j \leftarrow \frac{\hat{\ell}_j}{\max(\text{std}(\hat{\boldsymbol{\ell}}),\; 10^{-6})}$$

The concentration parameters are:

$$\alpha_j = \alpha_{\text{floor}} + \text{softplus}(\hat{\ell}_j \cdot \alpha_{\text{scale}}) + \epsilon$$

where $\text{softplus}(x) = \ln(1 + e^x)$, $\alpha_{\text{floor}} \geq 1.0$ ensures concentrations never fall below 1 (preventing degenerate U-shaped Dirichlet modes), and $\alpha_{\text{scale}}$ controls conviction spread.

#### 6.6.2 Alternative: `exp_tanh`

For comparison, the bounded exponential activation:

$$\alpha_j = \exp\!\big(\tanh(\ell_j) \cdot s\big) + \epsilon$$

where $s$ is a scale parameter. This produces $\alpha_j \in [\exp(-s) + \epsilon,\; \exp(s) + \epsilon]$.

| Scale $s$ | $\alpha$ range |
|-----------|---------------|
| 2.5 | $[0.08, 12.2]$ |
| 3.5 | $[0.03, 33.1]$ |

#### 6.6.3 Alpha Cap

A safety ceiling prevents extreme concentrations:

$$\alpha_j \leftarrow \min(\alpha_j, \alpha_{\text{cap}})$$

Final guarantee of strict positivity:

$$\alpha_j \leftarrow \max(\alpha_j, 10^{-6})$$

---

### 6.7 Dirichlet Distribution and Sampling

The portfolio allocation is drawn from a Dirichlet distribution parameterized by $\boldsymbol{\alpha}$:

$$\mathbf{w} \sim \text{Dir}(\boldsymbol{\alpha}), \quad \mathbf{w} \in \Delta^N$$

where $\Delta^N = \{\mathbf{w} \in \mathbb{R}^{N+1}_{\geq 0} : \sum_j w_j = 1\}$ is the probability simplex over $N$ risky assets plus cash.

**Probability density function:**

$$p(\mathbf{w} | \boldsymbol{\alpha}) = \frac{\Gamma\!\left(\sum_{j=1}^{N+1} \alpha_j\right)}{\prod_{j=1}^{N+1} \Gamma(\alpha_j)} \prod_{j=1}^{N+1} w_j^{\alpha_j - 1}$$

**Log-probability (used in PPO ratio computation):**

$$\log p(\mathbf{w} | \boldsymbol{\alpha}) = \log \Gamma\!\left(\textstyle\sum_j \alpha_j\right) - \sum_j \log \Gamma(\alpha_j) + \sum_j (\alpha_j - 1) \log w_j$$

**Expected allocation (deterministic mode):**

$$\mathbb{E}[w_j] = \frac{\alpha_j}{\alpha_0}, \quad \alpha_0 = \sum_{j=1}^{N+1} \alpha_j$$

**Variance:**

$$\text{Var}(w_j) = \frac{\alpha_j(\alpha_0 - \alpha_j)}{\alpha_0^2(\alpha_0 + 1)}$$

**Entropy:**

$$H[\text{Dir}(\boldsymbol{\alpha})] = \log B(\boldsymbol{\alpha}) + (\alpha_0 - N - 1)\psi(\alpha_0) - \sum_j (\alpha_j - 1)\psi(\alpha_j)$$

where $\psi(\cdot)$ is the digamma function and $B(\boldsymbol{\alpha}) = \frac{\prod_j \Gamma(\alpha_j)}{\Gamma(\alpha_0)}$.

**Interpretation of $\alpha_j$:**
- $\alpha_j > 1$: peaked mode at $w_j = (\alpha_j - 1)/(\alpha_0 - N - 1)$
- $\alpha_j = 1$: uniform marginal (no preference)
- $\alpha_j < 1$: boundary-seeking (degenerate, avoided by $\alpha_{\text{floor}}$)
- Higher $\alpha_0 = \sum_j \alpha_j$: lower variance → more deterministic policy

**Concentration ratio** (conviction metric):

$$\rho = \frac{\max_j \alpha_j}{\text{mean}_j \alpha_j}$$

Values $\rho > 1.5$ indicate meaningful asset discrimination.

## 7. Why the Shared-Backbone Design Matters
A key architectural point is that the model does **not** use three separate full agents.

### The design choice
Keep shared:
- temporal encoding
- cross-asset/context encoding
- regime-conditioned backbone

Separate:
- objective adapters
- actor heads
- critic heads
- router

### Why this is the right compromise
Fully separate agents would:
- multiply parameters
- reduce data efficiency
- make interpretation harder

Fully shared monolithic heads would:
- reintroduce objective interference
- encourage head collapse

The paper should explicitly argue that:
> the right granularity of separation is not the backbone, but the objective interface.

That is an actual methodological claim worth testing.


## 8. Reward Mapping and Curriculum

### 8.1 TAPE Reward Decomposition

The step-level reward is a weighted sum of three components aligned to the objective experts:

$$r_t = r^{\text{base}}_t + r^{\text{DSR}}_t + r^{\text{turn}}_t$$

A terminal bonus $r^{\text{term}}_T$ is added at the end of each episode.

**Expert-to-reward mapping:**

| Expert | Reward Component | Symbol |
|--------|-----------------|--------|
| Return | Base portfolio return | $r^{\text{base}}_t$ |
| Risk | Differential Sharpe Ratio (PBRS) | $r^{\text{DSR}}_t$ |
| Discipline | Turnover penalty (soft ceiling) | $r^{\text{turn}}_t$ |
| Blended (terminal) | TAPE score bonus | $r^{\text{term}}_T$ |

---

#### 8.1.1 Component $c_0$: Base Return

$$r^{\text{base}}_t = R^{\text{port}}_t \times 100 \times w_{\text{base}}$$

where $R^{\text{port}}_t$ is the net portfolio return at step $t$:

$$R^{\text{port}}_t = \sum_{j=1}^{N+1} w_{j,t} \cdot r_{j,t}$$

with $r_{j,t}$ being the individual asset return and $w_{j,t}$ the portfolio weight. The scaling factor of 100 converts fractional returns to percentage-scale rewards.

**Transaction costs** are subtracted from portfolio value before computing returns:

$$\text{TC}_t = c_{\text{rate}} \cdot \text{PV}_t \cdot \tau_t, \quad \tau_t = \sum_{j=1}^{N+1} |w_{j,t} - w_{j,t-1}|$$

where $c_{\text{rate}}$ is the cost rate (default: 1 basis point = $10^{-4}$) and $\tau_t$ is the portfolio turnover.

> **Example:** A portfolio of 10 stocks returns $+0.3\%$ on a given day. With $w_{\text{base}} = 1.0$:
> $$r^{\text{base}}_t = 0.003 \times 100 \times 1.0 = 0.30$$
> If turnover was $\tau_t = 0.15$ on a \$100K portfolio at $c_{\text{rate}} = 10^{-4}$:
> $$\text{TC}_t = 10^{-4} \times 100{,}000 \times 0.15 = \$1.50$$

---

#### 8.1.2 Component $c_1$: Differential Sharpe Ratio (PBRS)

The DSR component uses Potential-Based Reward Shaping (PBRS) with the rolling Sharpe ratio as the potential function. This preserves the optimal policy while providing dense risk-aware feedback.

**Rolling Sharpe ratio** over a window of $W$ steps:

$$\text{SR}_{t,W} = \frac{\bar{R}_W}{\hat{\sigma}_W} \cdot \sqrt{252}$$

where:

$$\bar{R}_W = \frac{1}{W}\sum_{\tau=t-W+1}^{t} R^{\text{port}}_\tau, \quad \hat{\sigma}_W = \sqrt{\frac{1}{W-1}\sum_{\tau=t-W+1}^{t} (R^{\text{port}}_\tau - \bar{R}_W)^2}$$

**PBRS formulation.** The DSR component uses the potential-based shaping theorem (Ng et al., 1999):

$$\Phi(s_t) = \gamma \cdot \text{SR}_{t,W}$$

$$r^{\text{DSR}}_t = (\Phi(s_t) - \Phi(s_{t-1})) \cdot \kappa_{\text{DSR}} \cdot w_{\text{DSR}}$$

which expands to:

$$r^{\text{DSR}}_t = (\gamma \cdot \text{SR}_{t,W} - \Phi_{\text{prev}}) \cdot \kappa_{\text{DSR}} \cdot w_{\text{DSR}}$$

where $\kappa_{\text{DSR}}$ is a scalar multiplier (default: 5.0), $w_{\text{DSR}}$ is the curriculum weight, and $\Phi_{\text{prev}}$ is updated to $\gamma \cdot \text{SR}_{t,W}$ after each step.

**Regime-conditional scaling.** The DSR is asymmetrically scaled by realized volatility regime:

$$\sigma^{\text{ann}}_{\text{recent}} = \text{std}(R_{t-W_v:t}) \cdot \sqrt{252}$$

$$r^{\text{DSR}}_t \leftarrow \begin{cases} r^{\text{DSR}}_t \cdot m^+_{\text{regime}} & \text{if } r^{\text{DSR}}_t \geq 0 \\ r^{\text{DSR}}_t \cdot m^-_{\text{regime}} & \text{if } r^{\text{DSR}}_t < 0 \end{cases}$$

| Regime | Condition | $(m^+, m^-)$ | Rationale |
|--------|-----------|--------------|-----------|
| Low vol | $\sigma^{\text{ann}} \leq \theta_{\text{low}}$ | $(1.0, 0.5)$ | Reduce penalty, encourage rotation |
| Mid vol | $\theta_{\text{low}} < \sigma^{\text{ann}} < \theta_{\text{high}}$ | $(1.0, 1.0)$ | Neutral |
| High vol | $\sigma^{\text{ann}} \geq \theta_{\text{high}}$ | $(1.5, 1.5)$ | Amplify both, stay defensive |

> **Example:** At step $t$, the rolling Sharpe (60-day window) is $\text{SR}_{t,60} = 1.2$ and $\Phi_{\text{prev}} = 1.15$. With $\gamma = 0.99$, $\kappa_{\text{DSR}} = 5.0$, $w_{\text{DSR}} = 0.5$:
> $$\Phi(s_t) = 0.99 \times 1.2 = 1.188$$
> $$r^{\text{DSR}}_t = (1.188 - 1.15) \times 5.0 \times 0.5 = 0.038 \times 2.5 = 0.095$$
> This small positive signal rewards the policy for improving its risk-adjusted performance.

---

#### 8.1.3 Component $c_2$: Turnover Penalty (Soft Ceiling)

The turnover penalty is a one-sided linear penalty that only activates when turnover exceeds a target ceiling $\tau^*$:

$$r^{\text{turn}}_t = \begin{cases} -\dfrac{\tau_t - \tau^*}{\max(\tau^*, 10^{-8})} \cdot \kappa_{\text{turn}} \cdot w_{\text{turn}} & \text{if } \tau_t > \tau^* \\[6pt] 0 & \text{if } \tau_t \leq \tau^* \end{cases}$$

where $\kappa_{\text{turn}}$ is a penalty scalar (default: 5.0), $w_{\text{turn}}$ is the curriculum weight, and $\tau^*$ is the per-step turnover target derived from the annualized target:

$$\tau^*_{\text{step}} = \frac{\tau^*_{\text{annual}}}{252}$$

> **Example:** Target annual turnover is $\tau^*_{\text{annual}} = 0.76$, giving $\tau^*_{\text{step}} = 0.76/252 \approx 0.003$. If actual turnover is $\tau_t = 0.08$ (a large rebalance), with $\kappa_{\text{turn}} = 5.0$ and $w_{\text{turn}} = 1.0$:
> $$r^{\text{turn}}_t = -\frac{0.08 - 0.003}{0.003} \times 5.0 \times 1.0 = -\frac{0.077}{0.003} \times 5.0 = -128.3$$
> This is then clipped by the final reward cap of $[-150, 150]$. The large penalty strongly discourages excessive trading.

---

#### 8.1.4 Terminal TAPE Bonus

At episode termination ($t = T$), a scalar bonus is computed from the Terminal Aggregate Performance Enhancement (TAPE) score:

$$\text{TAPE} = \sum_{i=1}^{M} \bar{w}_i \cdot U_i(m_i)$$

where $\bar{w}_i = w_i / \sum_j w_j$ are normalized component weights and $U_i$ is an asymmetric sigmoid utility for metric $m_i$:

$$U_i(x) = a_i + (b_i - a_i) \cdot \begin{cases} \dfrac{1}{1 + \exp(-k^+_i (x - \mu_i))} & \text{if } x \geq \mu_i \\[6pt] \dfrac{1}{1 + \exp(-k^-_i (x - \mu_i))} & \text{if } x < \mu_i \end{cases}$$

where $\mu_i$ is the target value, $k^+_i, k^-_i$ control steepness above and below target, and $[a_i, b_i]$ bound the utility output.

**TAPE metrics** (default configuration):

| Metric $m_i$ | Target $\mu_i$ | Direction |
|--------------|----------------|-----------|
| Sharpe ratio | 1.0 | increasing |
| Sortino ratio | 1.5 | increasing |
| Max drawdown | 0.15 | decreasing |
| Turnover | 0.8 | decreasing |
| Skewness | 0.0 | increasing |

**Terminal bonus modes:**

*Centered mode:*
$$r^{\text{term}}_T = (\text{TAPE} - b) \cdot \kappa_{\text{term}} \cdot w_{\text{term}}$$

*Signed mode (asymmetric):*
$$r^{\text{term}}_T = \begin{cases} \dfrac{\text{TAPE} - b}{1 - b} \cdot \kappa_{\text{term}} & \text{if TAPE} \geq b \\[6pt] -\dfrac{b - \text{TAPE}}{b} \cdot \kappa_{\text{term}} & \text{if TAPE} < b \end{cases}$$

where $b$ is the baseline threshold (e.g., 0.5).

**Gate A (safety floor).** If the episode Sharpe is too low or drawdown too severe, the bonus is forced non-positive:

$$\text{If } \text{SR}_{\text{ep}} \leq \theta_{\text{SR}} \;\text{ or }\; \text{MDD}_{\text{ep}} \geq \theta_{\text{MDD}}: \quad r^{\text{term}}_T \leftarrow -|r^{\text{term}}_T|$$

**Episode CVaR bonus.** An additional tail-risk component rewards staying within a regime-specific CVaR budget:

$$\text{CVaR}_\alpha = \frac{1}{\lfloor \alpha N \rfloor} \sum_{i=1}^{\lfloor \alpha N \rfloor} R_{(i)}$$

where $R_{(1)} \leq R_{(2)} \leq \ldots$ are the sorted episode returns. The bonus is:

$$r^{\text{CVaR}}_T = (\text{CVaR}_\alpha - \theta^{\text{regime}}_{\text{CVaR}}) \cdot \kappa_{\text{CVaR}}$$

> **Example:** An episode achieves Sharpe=1.3, Sortino=1.8, MDD=12%, Turnover=0.7, Skew=0.1. Suppose TAPE maps these to utilities $[0.72, 0.68, 0.75, 0.80, 0.55]$ with equal weights. Then:
> $$\text{TAPE} = \frac{1}{5}(0.72 + 0.68 + 0.75 + 0.80 + 0.55) = 0.70$$
> In centered mode with baseline $b = 0.5$ and $\kappa_{\text{term}} = 50$:
> $$r^{\text{term}}_T = (0.70 - 0.50) \times 50 = 10.0$$

---

#### 8.1.5 Final Reward Assembly

The total step reward, after all components and optional benchmark shaping:

$$r_t = \text{clip}\!\left(r^{\text{base}}_t + r^{\text{DSR}}_t + r^{\text{turn}}_t + r^{\text{bench}}_t,\; -150,\; 150\right)$$

At terminal step additionally:

$$r_T \leftarrow r_T + r^{\text{term}}_T \cdot w_{\text{term}}$$

**Reward normalization.** Before entering the PPO buffer, rewards are normalized using exponential moving average statistics:

$$\hat{r}_t = \text{clip}\!\left(\frac{r_t - \mu_{\text{EMA}}}{\max(\sigma_{\text{EMA}}, 0.1)},\; -5,\; 5\right)$$

where $\mu_{\text{EMA}}$ and $\sigma_{\text{EMA}}$ are updated after each batch with decay $\beta = 0.999$.

---

### 8.2 Reward Curriculum (Staged Expert Activation)

The reward weights and expert mask evolve through a staged curriculum indexed by training step $n$.

#### Phase A: Foundation (Return Only)

$$\begin{aligned}
\mathbf{m} &= [1, 0, 0] \\
w_{\text{base}} &= 1.0, \quad w_{\text{DSR}} = 0.0, \quad w_{\text{turn}} = 0.0, \quad w_{\text{term}} = 0.0
\end{aligned}$$

**Goal:** Learn concentrated alpha-seeking behavior without multi-objective interference.

> **Example:** At step 50K during Phase A, the agent is purely maximizing $r^{\text{base}}_t$. The router is forced to output $\mathbf{g} = [1, 0, 0]$ because only the return expert is unmasked. This produces the cleanest return-seeking signal.

#### Phase B: Risk Ramp

$$\begin{aligned}
\mathbf{m} &= [1, 1, 0] \\
w_{\text{DSR}}(n) &= \text{lerp}(0 \to 1.0,\; n_{\text{B,start}} \to n_{\text{B,end}})
\end{aligned}$$

where $\text{lerp}$ denotes linear interpolation over the step range.

**Goal:** Add risk awareness without destroying alpha structure. The smooth ramp prevents the catastrophic Phase B crashes observed in Run17-test.

> **Example:** At the midpoint of Phase B, $w_{\text{DSR}} = 0.5$. The router can now distribute weight between return and risk experts. If the market enters a high-volatility regime, the router may shift to $\mathbf{g} \approx [0.4, 0.6, 0]$, emphasizing risk management.

#### Phase C: Full TAPE

$$\begin{aligned}
\mathbf{m} &= [1, 1, 1] \\
w_{\text{turn}}(n) &= \text{lerp}(0 \to 1.0,\; n_{\text{C,start}} \to n_{\text{C,end}}) \\
w_{\text{term}}(n) &= \text{lerp}(0 \to 1.0,\; n_{\text{C,start}} \to n_{\text{C,end}})
\end{aligned}$$

**Goal:** Learn when to trade, hedge, or stay disciplined. All three experts are active and the router learns regime-dependent blending.

#### Smooth 7-Phase Schedule

The actual implementation uses a finer-grained 7-phase schedule with intermediate blend points:

| Phase | Steps | $w_{\text{base}}$ | $w_{\text{DSR}}$ | $w_{\text{turn}}$ | $w_{\text{term}}$ | Mask |
|-------|-------|-------------------|-------------------|--------------------|--------------------|------|
| A1 | $[0, n_1)$ | 1.0 | 0.0 | 0.0 | 0.0 | $[1,0,0]$ |
| A2 | $[n_1, n_2)$ | 1.0 | 0.25 | 0.0 | 0.0 | $[1,1,0]$ |
| B1 | $[n_2, n_3)$ | 1.0 | 0.55 | 0.0 | 0.0 | $[1,1,0]$ |
| B2 | $[n_3, n_4)$ | 1.0 | 0.80 | 0.25 | 0.0 | $[1,1,1]$ |
| B3 | $[n_4, n_5)$ | 1.0 | 1.0 | 0.55 | 0.25 | $[1,1,1]$ |
| C1 | $[n_5, n_6)$ | 1.0 | 1.0 | 0.80 | 0.55 | $[1,1,1]$ |
| C2 | $[n_6, \infty)$ | 1.0 | 1.0 | 1.0 | 1.0 | $[1,1,1]$ |

### 8.3 Why This Curriculum Matters

This is not only an optimization trick. It is part of the research methodology because it tests whether **objective separation plus staged activation** yields smoother transitions than a monolithic objective stack.

The curriculum addresses a fundamental failure mode in multi-objective portfolio RL: **gradient interference**. When return-seeking, risk-managing, and discipline-enforcing objectives are simultaneously active from step 0, their gradients can destructively interfere, leading to:

1. **Alpha collapse**: all Dirichlet concentrations converge to $\alpha_j \approx 1$ (uniform)
2. **Turnover spikes**: the policy oscillates rapidly between conflicting objectives
3. **Training instability**: critic loss fails to converge

The staged curriculum ensures that each expert develops meaningful specialization before competing objectives are introduced.

> **Example of failure without staging (Run17-test):** Phase A (return-only) achieved Sharpe 1.46 with healthy alpha diversity (mean $\alpha \approx 1.75$, NVDA $\alpha \approx 3.0$). At step 30K, DSR was abruptly activated at full weight. Within 5K steps: turnover spiked from 20% to 36%, alphas collapsed to $\approx 1.0$, and final Sharpe was $-1.05$.

## 9. Loss Design

The total loss optimized by ROE-TAPE combines six components:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{PPO}} + \mathcal{L}_{\text{critic}} + \mathcal{L}_{\text{entropy}} + \mathcal{L}_{\text{expert}} + \mathcal{L}_{\text{diversity}} + \mathcal{L}_{\text{router}}$$

---

### 9.1 PPO Clipped Surrogate Loss (Actor)

The primary policy optimization uses the PPO-Clip objective (Schulman et al., 2017).

**Probability ratio.** For the Dirichlet policy with parameters $\boldsymbol{\alpha}_\theta$ (current) and $\boldsymbol{\alpha}_{\theta_{\text{old}}}$ (rollout):

$$\rho_t(\theta) = \frac{\pi_\theta(\mathbf{w}_t | s_t)}{\pi_{\theta_{\text{old}}}(\mathbf{w}_t | s_t)} = \exp\!\big(\log \pi_\theta(\mathbf{w}_t | s_t) - \log \pi_{\theta_{\text{old}}}(\mathbf{w}_t | s_t)\big)$$

where the Dirichlet log-probability is:

$$\log \pi_\theta(\mathbf{w} | s) = \log \Gamma\!\left(\sum_j \alpha_j\right) - \sum_j \log \Gamma(\alpha_j) + \sum_j (\alpha_j - 1) \log w_j$$

**Safety clipping.** The log-ratio is double-clipped to prevent numerical instability:

$$\Delta_{\log} = \text{clip}(\log \pi_\theta - \log \pi_{\theta_{\text{old}}},\; -10,\; 10)$$

$$\rho_t = \exp\!\left(\text{clip}\!\left(\Delta_{\log},\; \log(1-\epsilon),\; \log(1+\epsilon)\right)\right)$$

where $\epsilon$ is the PPO clip parameter (typically 0.2).

**Clipped surrogate objective:**

$$\mathcal{L}_{\text{PPO}} = -\frac{1}{B}\sum_{t=1}^{B} \min\!\left(\rho_t \hat{A}_t,\; \text{clip}(\rho_t, 1-\epsilon, 1+\epsilon) \hat{A}_t\right)$$

where $B$ is the minibatch size and $\hat{A}_t$ is the GAE advantage estimate.

> **Example:** Suppose the old policy gave $\log \pi_{\text{old}} = -5.2$ and the new policy gives $\log \pi_{\text{new}} = -4.8$ for a particular action. Then $\rho = \exp(0.4) \approx 1.49$. If the advantage $\hat{A}_t = 0.5$ and $\epsilon = 0.2$:
> - Unclipped: $\rho \hat{A} = 1.49 \times 0.5 = 0.745$
> - Clipped: $\text{clip}(1.49, 0.8, 1.2) \times 0.5 = 1.2 \times 0.5 = 0.6$
> - PPO takes $\min(0.745, 0.6) = 0.6$ — the clip prevents the policy from moving too far.

---

### 9.2 Generalized Advantage Estimation (GAE)

Advantages are computed using GAE (Schulman et al., 2016) with discount $\gamma$ and trace decay $\lambda$:

$$\delta_t = r_t + \gamma V(s_{t+1})(1 - d_t) - V(s_t)$$

$$\hat{A}_t = \sum_{\ell=0}^{T-t-1} (\gamma \lambda)^\ell (1 - d_{t+\ell}) \cdot \delta_{t+\ell}$$

which is computed efficiently via the backward recursion:

$$\hat{A}_t = \delta_t + \gamma \lambda (1 - d_t) \hat{A}_{t+1}$$

with $\hat{A}_T = 0$. The value targets (returns) are:

$$G_t = \hat{A}_t + V(s_t)$$

**Per-expert GAE.** When separate critics are used, each expert has its own advantage stream:

$$\delta^{(k)}_t = r^{(k)}_t + \gamma V_k(s_{t+1})(1 - d_t) - V_k(s_t)$$

$$\hat{A}^{(k)}_t = \sum_{\ell=0}^{T-t-1} (\gamma \lambda)^\ell (1-d_{t+\ell}) \cdot \delta^{(k)}_{t+\ell}$$

where $r^{(k)}_t$ is the reward component for expert $k$ and $V_k$ is the expert-specific critic.

> **Example:** Consider a 3-step trajectory with $\gamma = 0.99$, $\lambda = 0.95$:
> - Step 2 (last): $\delta_2 = r_2 + 0 - V(s_2) = 0.5 - 0.3 = 0.2$, so $\hat{A}_2 = 0.2$
> - Step 1: $\delta_1 = r_1 + 0.99 \times V(s_2) - V(s_1) = 0.1 + 0.297 - 0.4 = -0.003$
>   - $\hat{A}_1 = -0.003 + 0.99 \times 0.95 \times 0.2 = -0.003 + 0.188 = 0.185$
> - Step 0: $\delta_0 = r_0 + 0.99 \times V(s_1) - V(s_0) = 0.3 + 0.396 - 0.5 = 0.196$
>   - $\hat{A}_0 = 0.196 + 0.99 \times 0.95 \times 0.185 = 0.196 + 0.174 = 0.370$

---

### 9.3 Critic Loss

#### 9.3.1 Scalar Critic (Blended)

The blended critic is trained with normalized MSE loss with optional value clipping:

$$\hat{V}_{\text{norm}} = \frac{V(s_t) - \mu_G}{\sigma_G + \epsilon}, \quad \hat{G}_{\text{norm}} = \frac{G_t - \mu_G}{\sigma_G + \epsilon}$$

**Without value clipping:**

$$\mathcal{L}_{\text{critic}} = \frac{1}{B}\sum_{t=1}^{B} (\hat{G}_{\text{norm},t} - \hat{V}_{\text{norm},t})^2$$

**With value clipping** (prevents catastrophic value updates):

$$V_{\text{clip}}(s_t) = V_{\text{old}}(s_t) + \text{clip}(V(s_t) - V_{\text{old}}(s_t),\; -\epsilon_v,\; \epsilon_v)$$

$$\mathcal{L}_{\text{critic}} = \frac{1}{B}\sum_{t=1}^{B} \max\!\left((\hat{G}_t - \hat{V}_t)^2,\; (\hat{G}_t - \hat{V}_{\text{clip},t})^2\right)$$

where $\epsilon_v$ is the value clip range.

#### 9.3.2 Expert Critics (Separate)

Each expert critic $V_k$ is trained on its objective-specific returns $G^{(k)}_t$:

$$\mathcal{L}_{\text{critic}}^{(k)} = \frac{1}{B}\sum_{t=1}^{B} (G^{(k)}_t - V_k(s_t))^2$$

The total expert critic loss, masked by active experts:

$$\mathcal{L}_{\text{expert-critic}} = \frac{\sum_k m_k \sum_{t=1}^B (G^{(k)}_t - V_k(s_t))^2}{B \cdot \sum_k m_k}$$

where $m_k \in \{0, 1\}$ is the expert mask.

---

### 9.4 Entropy Regularization

The policy entropy encourages exploration:

$$\mathcal{L}_{\text{entropy}} = -c_{\text{ent}} \cdot \frac{1}{B}\sum_{t=1}^{B} H[\text{Dir}(\boldsymbol{\alpha}_t)]$$

where $c_{\text{ent}}$ is the entropy coefficient (annealed during training, e.g., $0.01 \to 0.001$).

The Dirichlet entropy $H[\text{Dir}(\boldsymbol{\alpha})]$ is given in Section 6.7.

> **Example:** With $\boldsymbol{\alpha} = [2.0, 3.0, 1.5, 2.5]$ (4 assets), $\alpha_0 = 9.0$. The entropy is relatively high, indicating a broad distribution. As training progresses and $\boldsymbol{\alpha}$ concentrations increase (e.g., to $[5, 8, 3, 6]$, $\alpha_0 = 22$), the entropy decreases, reflecting a more deterministic policy.

---

### 9.5 Objective Expert Auxiliary Loss

The expert auxiliary loss trains each expert head to produce policies aligned with its objective-specific advantages:

$$\mathcal{L}_{\text{expert}} = -\frac{c_{\text{aux}}}{B \cdot |\mathcal{K}_{\text{active}}|} \sum_{k \in \mathcal{K}_{\text{active}}} \sum_{t=1}^{B} \hat{A}^{(k)}_t \cdot \log \pi^{(k)}_\theta(\mathbf{w}_t | s_t)$$

where:
- $c_{\text{aux}}$ is the auxiliary loss coefficient
- $\mathcal{K}_{\text{active}} = \{k : m_k = 1\}$ are the currently active experts
- $\pi^{(k)}_\theta(\mathbf{w} | s) = \text{Dir}(\mathbf{w}; \boldsymbol{\alpha}_k)$ is the expert-specific Dirichlet distribution
- $\hat{A}^{(k)}_t$ is the advantage from the expert-specific critic (stop-gradient applied)

This loss encourages each expert to independently improve on its own objective, preventing expert collapse where all heads learn identical policies.

> **Example:** During Phase A ($\mathbf{m} = [1,0,0]$), only the return expert contributes. If the return expert's advantage $\hat{A}^{(\text{ret})}_t = 2.0$ for an action that increased portfolio value, the loss pushes the return expert to increase the probability of that action. The risk and discipline experts receive no gradient signal yet.

---

### 9.6 Expert Diversity Loss

The diversity loss penalizes similarity between expert policies, preventing expert collapse:

$$\mathcal{L}_{\text{diversity}} = c_{\text{div}} \cdot \frac{1}{B} \sum_{t=1}^{B} \bar{S}_t$$

where $\bar{S}_t$ is the mean pairwise cosine similarity between active experts:

$$\bar{S}_t = \frac{\sum_{j \neq k} m_j m_k \cdot \cos(\hat{\boldsymbol{\alpha}}^{(j)}_t, \hat{\boldsymbol{\alpha}}^{(k)}_t)}{\sum_{j \neq k} m_j m_k}$$

The normalized expert mean allocations are:

$$\hat{\boldsymbol{\alpha}}^{(k)} = \frac{\boldsymbol{\alpha}^{(k)} / \|\boldsymbol{\alpha}^{(k)}\|_1}{\|\boldsymbol{\alpha}^{(k)} / \|\boldsymbol{\alpha}^{(k)}\|_1\|_2}$$

That is, each expert's alpha is first converted to a mean allocation vector $\boldsymbol{\alpha}^{(k)} / \sum_j \alpha^{(k)}_j$, then L2-normalized before computing cosine similarity.

> **Example:** Suppose the return expert produces mean allocation $[0.15, 0.25, 0.10, 0.30, 0.20]$ and the risk expert produces $[0.20, 0.20, 0.20, 0.20, 0.20]$ (equal-weight, defensive). The cosine similarity would be moderate ($\sim 0.85$). But if both experts produce identical allocations, $\cos = 1.0$ and the diversity loss is maximized, pushing them to differentiate.

---

### 9.7 Router Entropy Regularization

The router entropy loss encourages the router to use all active experts rather than collapsing to a single expert:

$$\mathcal{L}_{\text{router}} = -c_{\text{rent}} \cdot \frac{1}{B}\sum_{t=1}^{B} H[\mathbf{g}_t]$$

where:

$$H[\mathbf{g}_t] = -\sum_{k=1}^{K} g_{k,t} \log \max(g_{k,t},\; 10^{-8})$$

Note the negative sign on $c_{\text{rent}}$: this term **maximizes** router entropy (adding $-c \cdot (-H) = c \cdot H$ to the objective being minimized).

> **Example:** If $\mathbf{g} = [0.9, 0.05, 0.05]$, the entropy is low: $H = -(0.9 \log 0.9 + 0.05 \log 0.05 + 0.05 \log 0.05) \approx 0.47$. With uniform routing $\mathbf{g} = [0.33, 0.33, 0.33]$, entropy is maximal: $H = \log 3 \approx 1.10$. The router entropy loss pushes toward the latter when active.

---

### 9.8 Alpha Diversity Loss (HHI Penalty)

An additional concentration diversity penalty based on the Herfindahl-Hirschman Index (HHI) discourages extreme portfolio concentration:

$$\mathcal{L}_{\text{HHI}} = c_{\text{HHI}} \cdot \frac{1}{B}\sum_{t=1}^{B} \text{HHI}(\hat{\mathbf{w}}_t)$$

where the expected portfolio weights are:

$$\hat{w}_{j,t} = \frac{\alpha_{j,t}}{\sum_k \alpha_{k,t}}, \quad \text{HHI}(\hat{\mathbf{w}}) = \sum_j \hat{w}_{j,t}^2$$

For $N+1$ assets, HHI ranges from $1/(N+1)$ (equal weight) to $1.0$ (single-asset concentration).

> **Example:** With 11 assets (10 + cash), equal weight gives $\text{HHI} = 11 \times (1/11)^2 = 1/11 \approx 0.091$. A portfolio concentrated 50% in one stock with 5% in each of the other 10 gives $\text{HHI} = 0.25 + 10 \times 0.0025 = 0.275$.

---

### 9.9 Approximate KL Divergence (Early Stopping)

The approximate KL divergence between old and new policies is monitored for early stopping of PPO epochs:

$$\hat{D}_{\text{KL}} = \frac{1}{B}\sum_{t=1}^{B} (\log \pi_{\theta_{\text{old}}}(\mathbf{w}_t | s_t) - \log \pi_\theta(\mathbf{w}_t | s_t))$$

If $\hat{D}_{\text{KL}} > \text{KL}_{\text{target}}$ (typically $1.5 \times \text{KL}_{\text{target-base}}$), the PPO update epoch is terminated early.

---

### 9.10 Complete Loss Assembly

The total loss minimized by the optimizer:

$$\boxed{\mathcal{L}_{\text{total}} = \underbrace{\mathcal{L}_{\text{PPO}}}_{\text{policy}} + \underbrace{c_v \cdot \mathcal{L}_{\text{critic}}}_{\text{value}} + \underbrace{\mathcal{L}_{\text{entropy}}}_{\text{exploration}} + \underbrace{\mathcal{L}_{\text{expert}}}_{\text{specialization}} + \underbrace{\mathcal{L}_{\text{diversity}}}_{\text{anti-collapse}} + \underbrace{\mathcal{L}_{\text{router}}}_{\text{routing balance}} + \underbrace{\mathcal{L}_{\text{HHI}}}_{\text{concentration}}}$$

where $c_v$ is the value loss coefficient (typically 0.5).

**Gradient flow diagram:**

```
Shared TCN Encoder ← ∂L_PPO/∂θ + ∂L_expert/∂θ + ∂L_entropy/∂θ + ∂L_HHI/∂θ
Expert Adapters    ← ∂L_PPO/∂θ_k + ∂L_expert/∂θ_k + ∂L_diversity/∂θ_k
Expert Critics     ← ∂L_critic^(k)/∂φ_k  (no gradient to encoder)
Router MLP         ← ∂L_PPO/∂θ_r + ∂L_router/∂θ_r
```

Note: Expert advantages $\hat{A}^{(k)}$ are stop-gradiented when entering $\mathcal{L}_{\text{expert}}$ to prevent the expert loss from interfering with critic training.

## 10. Numerical Stability as a Research Engineering Contribution

The current codebase has already exposed a real issue: Dirichlet policies with low-concentration regions and boundary-adjacent actions can produce unstable log-probs during PPO updates. This should be documented as a methodological point, not hidden.

---

### 10.1 The Dirichlet Instability Problem

The Dirichlet log-probability contains two sources of numerical fragility:

**1. Log-Gamma singularities.** For small $\alpha_j$:

$$\log \Gamma(\alpha_j) \to -\infty \quad \text{as} \quad \alpha_j \to 0^+$$

This produces $\pm\infty$ terms in the log-prob that can cancel imprecisely in floating point.

**2. Boundary log-actions.** For actions near the simplex boundary where $w_j \approx 0$:

$$(\alpha_j - 1) \log w_j \to \begin{cases} -\infty & \text{if } \alpha_j > 1 \\ +\infty & \text{if } \alpha_j < 1 \end{cases}$$

> **Example:** Consider $\boldsymbol{\alpha} = [1.01, 1.01, \ldots, 1.01]$ (11-dim, near-uniform) and an action $\mathbf{w}$ where $w_3 = 10^{-7}$. Then $(\alpha_3 - 1)\log w_3 = 0.01 \times (-16.1) = -0.161$, which is manageable. But if $\alpha_3 = 0.5$ (sub-unity), $(\alpha_3 - 1)\log w_3 = -0.5 \times (-16.1) = +8.05$, dominating the entire log-prob and creating severe gradient distortion.

---

### 10.2 Stabilization Measures

#### 10.2.1 Alpha Floor Enforcement

The $\alpha_{\text{floor}}$ parameter in `cross_softplus` guarantees $\alpha_j \geq \alpha_{\text{floor}} + \epsilon$:

$$\alpha_j = \alpha_{\text{floor}} + \text{softplus}(\hat{\ell}_j \cdot \alpha_{\text{scale}}) + \epsilon \geq \alpha_{\text{floor}} + \epsilon > 1.0$$

With $\alpha_{\text{floor}} = 1.0$, concentrations never enter the sub-unity regime, eliminating the boundary-seeking pathology entirely.

> **Example:** Even if the network outputs $\hat{\ell}_j = -100$ (extreme negative logit), $\text{softplus}(-100 \times 2.5) \approx 0$, so $\alpha_j = 1.0 + 0 + 10^{-6} = 1.000001$. The distribution remains well-behaved.

#### 10.2.2 Simplex Stabilization Before Log-Prob

Before evaluating $\log \pi(\mathbf{w} | \boldsymbol{\alpha})$, actions are projected onto the interior of the simplex:

$$w_j \leftarrow \max(w_j, \epsilon_w), \quad \mathbf{w} \leftarrow \frac{\mathbf{w}}{\sum_j w_j}$$

where $\epsilon_w \approx 10^{-6}$. This prevents $\log w_j = -\infty$.

#### 10.2.3 Log-Prob Sanitization

The computed log-probabilities are sanitized before entering PPO:

$$\log \pi \leftarrow \begin{cases} \log \pi & \text{if } |\log \pi| < \infty \\ 0 & \text{otherwise (non-finite)} \end{cases}$$

The log-ratio is further bounded:

$$\Delta_{\log} = \text{clip}(\log \pi_{\text{new}} - \log \pi_{\text{old}},\; -10,\; 10)$$

#### 10.2.4 Alpha Cap

The alpha cap prevents extreme concentrations that could cause numerical overflow in $\Gamma(\alpha_0)$ where $\alpha_0 = \sum_j \alpha_j$:

$$\alpha_j \leftarrow \min(\alpha_j, \alpha_{\text{cap}})$$

For $N = 10$ with $\alpha_{\text{cap}} = 50$: $\alpha_0 \leq 11 \times 50 = 550$. Since $\log \Gamma(550) \approx 3170$, this remains well within float64 precision.

> **Example:** Without the cap, if one expert head drives a logit to extreme values, $\alpha_j$ could reach thousands. With $\alpha_0 = 5000$: $\log \Gamma(5000) \approx 37{,}000$, still representable but producing very steep gradients that destabilize training.

#### 10.2.5 Non-Finite Guards in Gradient Path

The PPO loss computation includes explicit non-finite guards:

$$\mathcal{L}_{\text{PPO}} = -\frac{1}{B}\sum_{t} \mathbb{1}[\text{is\_finite}(\min(\text{surr}_1, \text{surr}_2))] \cdot \min(\text{surr}_1, \text{surr}_2)$$

Non-finite samples are replaced with zero, contributing no gradient signal rather than corrupting the entire batch.

---

### 10.3 Training Diagnostics

The following diagnostics are logged at each training step to monitor numerical health:

| Diagnostic | Formula | Healthy Range |
|-----------|---------|--------------|
| Alpha mean | $\bar{\alpha} = \frac{1}{N+1}\sum_j \alpha_j$ | $[1.5, 10.0]$ |
| Alpha std | $\text{std}(\boldsymbol{\alpha})$ | $> 0.1$ (diversity) |
| Alpha max | $\max_j \alpha_j$ | $< \alpha_{\text{cap}}$ |
| Concentration ratio | $\rho = \max_j \alpha_j / \bar{\alpha}$ | $> 1.3$ |
| Cap-hit fraction | $\frac{1}{B(N+1)}\sum_{t,j} \mathbb{1}[\alpha_{j,t} = \alpha_{\text{cap}}]$ | $< 0.05$ |
| Non-finite count | $\sum_t \mathbb{1}[\neg\text{is\_finite}(\log \pi_t)]$ | $= 0$ |
| Clip fraction | $\frac{1}{B}\sum_t \mathbb{1}[|\rho_t - 1| > \epsilon]$ | $[0.05, 0.30]$ |
| Approx KL | $\hat{D}_{\text{KL}}$ | $< 0.03$ |

> **Example of instability detection:** If non-finite count jumps from 0 to $> 10$ in a single batch, this indicates a pathological action or alpha configuration. The system logs the offending $\boldsymbol{\alpha}$ and $\mathbf{w}$ values for debugging. If cap-hit fraction exceeds 20%, the alpha activation is saturating and the policy cannot express further conviction — a signal to increase $\alpha_{\text{cap}}$ or switch activations.

### 10.4 Why This Matters for the Paper

If the paper uses simplex-native RL policies, reviewers will care about trainability. A publishable paper should show not only performance but also how the method remains numerically operable. The stability measures described above should be reported as:

1. **Empirical necessity**: without them, training fails in $> 30\%$ of seeds
2. **Zero overhead**: all guards are $O(B)$ elementwise operations
3. **Completeness**: the combination of floor + cap + sanitization + simplex projection covers all known failure modes

## 11. Primary Hypotheses
### H1. Objective separation improves stability
Compared with a monolithic Run18-style policy, ROE-TAPE should:
- reduce destructive transition effects at B/C stage boundaries
- improve training survivability
- reduce catastrophic turnover spikes

### H2. Separate critics improve specialization
Compared with separate actor heads but a shared critic, separate critics should:
- improve objective alignment
- reduce cross-objective interference
- improve worst-regime performance

### H3. Regime routing improves robustness
Compared with static expert averaging, learned routing should improve:
- worst-regime Sharpe
- consistency across market regimes
- drawdown behavior in difficult periods

### H4. Covariance summaries plus PC loadings improve cross-asset reasoning
Compared with eigenvalue-only covariance inputs, the full covariance summary set should improve:
- regime awareness
- robustness under correlation shifts
- defensive behavior in stressed regimes

### H5. Simplex-stable cross-softplus improves trainability over more brittle parameterizations
Compared with less controlled alternatives, strengthened cross-softplus with stability guards should:
- maintain alpha separation
- avoid cap saturation
- reduce catastrophic numerical failure


## 12. Experimental Methodology
### 12.1 Train/Test split
Keep the fixed time split already used in the codebase:
- train through `2019-12-31`
- test from `2020-01-02` onward

### 12.2 Training protocol
For all main experiments, keep constant:
- architecture backbone
- feature set
- sequence length
- PPO schedules unless the ablation is explicitly about them
- random seed protocol

### 12.3 Evaluation protocol
Use three layers of evaluation:

#### Layer 1: Full deterministic evaluation
- full test period
- standard metrics: Sharpe, return, MDD, volatility, turnover

#### Layer 2: Regime-stratified deterministic evaluation
- sweep fixed horizon windows across predefined regimes
- score by mean and worst-regime robustness

#### Layer 3: Regime-stratified stochastic confirmation
- for the selected winner, run stochastic confirmation within the same regime windows
- report mean and variability by regime

### 12.4 Multi-seed protocol
For publication quality, at least:
- `3` seeds for exploratory comparisons
- `5` seeds for the final primary claims if compute permits

A single-seed result should never be used as the only evidence for a contribution claim.


## 13. Required Ablation Program
The ablations are essential. Without them, the paper will read like a bundle of simultaneous changes.

### A. Architecture ablations
1. Run18 monolithic baseline
2. Shared backbone + separate actor heads + shared critic
3. Shared backbone + separate actor heads + separate critics, no router
4. Shared backbone + separate actor heads + separate critics + router
5. Full ROE-TAPE with expert adapters
6. Full ROE-TAPE without expert adapters

### B. Reward/curriculum ablations
1. abrupt component transitions
2. smooth staged transitions
3. no terminal bonus
4. no benchmark shaping
5. no discipline expert activation

### C. Router ablations
1. fixed uniform blending
2. phase mask only, no learned routing
3. learned routing with entropy regularization
4. learned routing without entropy regularization

### D. Critic ablations
1. separate critics
2. shared critic
3. separate critics without expert-specific advantages

### E. Policy/activation ablations
1. `cross_softplus` scale `2.5`
2. `cross_softplus` scale `3.0`
3. `cross_softplus` scale `3.5`
4. `exp_tanh` reference baseline
5. with vs without Dirichlet stability hardening

### F. Feature/covariance ablations
1. eigenvalues only
2. eigenvalues + explained variance ratios
3. + correlation structure summaries
4. + PC loadings (full)
5. reduced local feature set vs expanded local feature set

### G. Universe ablations
1. current balanced list
2. best-cap list
3. tech list

These should be treated as controlled universe ablations with the rest of Run19 fixed.


In [ ]:
import pandas as pd

ablation_rows = [
    ("A1", "Architecture", "Run18 monolithic baseline", "tests whether experts are needed at all"),
    ("A2", "Architecture", "Expert actors + shared critic", "isolates critic separation effect"),
    ("A3", "Architecture", "Expert actors + expert critics, no router", "isolates routing effect"),
    ("A4", "Architecture", "Full ROE-TAPE", "full proposed method"),
    ("B1", "Curriculum", "Abrupt transitions", "tests whether staged activation matters"),
    ("B2", "Curriculum", "Smooth transitions", "canonical curriculum"),
    ("C1", "Router", "Uniform expert averaging", "tests learned routing value"),
    ("C2", "Router", "Masked learned router", "canonical router"),
    ("D1", "Critic", "Shared critic", "tests separate critic necessity"),
    ("D2", "Critic", "Separate critics", "canonical critic design"),
    ("E1", "Activation", "cross_softplus scale 2.5", "weaker conviction"),
    ("E2", "Activation", "cross_softplus scale 3.0", "current preferred setting"),
    ("E3", "Activation", "cross_softplus scale 3.5", "higher-conviction stress test"),
    ("F1", "Covariance", "Eigenvalues only", "compressed market structure baseline"),
    ("F2", "Covariance", "Full covariance summary + PC loadings", "current preferred signal set"),
    ("G1", "Universe", "Current balanced 10-stock list", "reference universe"),
    ("G2", "Universe", "Best-cap 10-stock list", "robustness on a cleaner universe"),
    ("G3", "Universe", "Tech 10-stock list", "sector concentration stress test"),
]

ablation_df = pd.DataFrame(ablation_rows, columns=["id", "family", "variant", "purpose"])
ablation_df


## 14. Universe Study Design
A specific sub-study should compare universes while holding the method fixed.

### U1. Balanced baseline universe
Use the current balanced 10-stock list as the main reference.

### U2. Best-cap universe
Use a top-cap, high-liquidity list to test whether cleaner macro and correlation structure improves stability.

### U3. Tech universe
Use a tighter sector universe to test:
- concentration behavior
- higher-correlation dynamics
- whether routing and discipline matter more when diversification is limited

### Why this belongs in the paper
This gives the work a stronger external validity section:
- the method is not tested on only one arbitrarily chosen asset basket
- the paper can show whether objective routing is robust across different universe structures


## 15. Metrics That Must Be Reported

### 15.1 Performance Metrics

All metrics below are computed on the out-of-sample test period (post-2020-01-01).

**Total return:**

$$R_{\text{total}} = \frac{\text{PV}_T}{\text{PV}_0} - 1$$

**Annualized return:**

$$R_{\text{ann}} = \left(\frac{\text{PV}_T}{\text{PV}_0}\right)^{252/T} - 1$$

> **Example:** A portfolio grows from \$100K to \$180K over 1260 trading days (5 years): $R_{\text{total}} = 0.80$, $R_{\text{ann}} = 1.80^{252/1260} - 1 = 1.80^{0.2} - 1 \approx 0.125$ (12.5% annualized).

**Sharpe ratio** (excess return per unit risk, assuming $r_f = 0$):

$$\text{SR} = \frac{\bar{R}_{\text{daily}}}{\sigma_{\text{daily}}} \cdot \sqrt{252}$$

where $\bar{R}_{\text{daily}} = \frac{1}{T}\sum_{t=1}^T R_t$ and $\sigma_{\text{daily}} = \sqrt{\frac{1}{T-1}\sum_{t=1}^T (R_t - \bar{R})^2}$.

> **Example:** Daily mean return = 0.05%, daily std = 1.2%: $\text{SR} = \frac{0.0005}{0.012} \times \sqrt{252} = 0.0417 \times 15.87 = 0.66$.

**Sortino ratio** (penalizes only downside volatility):

$$\text{SoR} = \frac{\bar{R}_{\text{daily}}}{\sigma_{\text{down}}} \cdot \sqrt{252}, \quad \sigma_{\text{down}} = \sqrt{\frac{1}{T}\sum_{t=1}^T \min(R_t, 0)^2}$$

> **Example:** Same mean return 0.05%, but downside deviation = 0.8% (less than total std because positive returns don't count): $\text{SoR} = \frac{0.0005}{0.008} \times 15.87 = 0.99$.

**Maximum drawdown:**

$$\text{MDD} = \max_{0 \leq t_1 \leq t_2 \leq T} \frac{\text{PV}_{t_1} - \text{PV}_{t_2}}{\text{PV}_{t_1}}$$

> **Example:** Peak portfolio value was \$150K, trough was \$120K: $\text{MDD} = (150 - 120)/150 = 20\%$.

**Annualized volatility:**

$$\sigma_{\text{ann}} = \sigma_{\text{daily}} \cdot \sqrt{252}$$

**Portfolio turnover** (annualized):

$$\tau_{\text{ann}} = \frac{252}{T}\sum_{t=1}^T \sum_{j=1}^{N+1} |w_{j,t} - w_{j,t-1}|$$

> **Example:** Average daily turnover of 3% across all assets: $\tau_{\text{ann}} = 0.03 \times 252 = 7.56$ (756% annualized — very high, suggesting excessive trading).

---

### 15.2 Robustness Metrics

**Regime mean Sharpe.** Given $R$ predefined regime windows $\{(t^r_{\text{start}}, t^r_{\text{end}})\}_{r=1}^R$:

$$\overline{\text{SR}}_{\text{regime}} = \frac{1}{R}\sum_{r=1}^R \text{SR}_r$$

**Worst-regime Sharpe:**

$$\text{SR}_{\text{worst}} = \min_{r \in \{1, \ldots, R\}} \text{SR}_r$$

**Regime consistency (dispersion):**

$$\text{CV}_{\text{regime}} = \frac{\text{std}(\{\text{SR}_r\})}{\overline{\text{SR}}_{\text{regime}}}$$

Lower $\text{CV}_{\text{regime}}$ indicates more consistent performance across market conditions.

> **Example:** A model achieves Sharpe by regime: Bull=1.8, Sideways=0.9, Bear=-0.2, Recovery=1.1. Then $\overline{\text{SR}} = 0.9$, $\text{SR}_{\text{worst}} = -0.2$, $\text{CV}_{\text{regime}} = 0.74/0.9 = 0.82$. A reviewer would note the model struggles in bear markets.

**Stochastic regime variability.** For stochastic evaluation with $S$ seeds per regime:

$$\text{Var}_{\text{stoch}}^{(r)} = \text{std}(\{\text{SR}^{(r)}_s\}_{s=1}^S)$$

---

### 15.3 Training Stability Metrics

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| Non-finite incidents | $\sum_{\text{batch}} \mathbb{1}[\exists t: \neg\text{finite}(\log\pi_t)]$ | Should be 0; indicates numerical failure |
| Critic loss | $\mathcal{L}_{\text{critic}}$ trajectory | Should decrease monotonically |
| Actor loss | $\mathcal{L}_{\text{PPO}}$ trajectory | Stable oscillation around 0 |
| KL divergence | $\hat{D}_{\text{KL}}$ per update | Should stay $< 0.03$ |
| Clip fraction | $\frac{1}{B}\sum_t \mathbb{1}[\rho_t \text{ clipped}]$ | $0.05$-$0.30$ is healthy |
| Gradient norm | $\|\nabla_\theta \mathcal{L}\|_2$ | Sudden spikes indicate instability |
| Alpha range | $[\min_j \alpha_j, \max_j \alpha_j]$ | Should show spread, not collapse |
| Alpha std | $\text{std}(\boldsymbol{\alpha})$ | $> 0.1$ indicates diversity |
| Cap-hit fraction | See Section 10.3 | $< 5\%$ indicates headroom |

---

### 15.4 Expert Behavior Metrics

**Average router weights by regime.** For regime $r$:

$$\bar{\mathbf{g}}_r = \frac{1}{|r|}\sum_{t \in r} \mathbf{g}_t$$

> **Example (ideal behavior):** In a bull regime, $\bar{\mathbf{g}}_{\text{bull}} = [0.7, 0.1, 0.2]$ (return-dominated). In a crisis, $\bar{\mathbf{g}}_{\text{crisis}} = [0.2, 0.6, 0.2]$ (risk-dominated). This shows the router has learned regime-appropriate blending.

**Router entropy by phase:**

$$\bar{H}_{\text{phase}} = \frac{1}{|\text{phase}|}\sum_{t \in \text{phase}} H[\mathbf{g}_t]$$

Should increase from Phase A ($H \approx 0$ because only one expert is active) to Phase C ($H > 0$ if the router uses multiple experts).

**Expert diversity score:**

$$D_{\text{expert}} = 1 - \bar{S}_t$$

where $\bar{S}_t$ is the mean pairwise cosine similarity (Section 9.6). $D = 0$ means identical experts, $D = 1$ means fully orthogonal.

**Objective-specific critic trajectories.** Each critic's value estimate $V_k(s_t)$ should track its respective reward stream, with the return critic showing higher variance than the discipline critic.

A reviewer should be able to see not only that the method performs, but **how** it behaves — which expert dominates when, how the router responds to regime shifts, and whether the critics accurately decompose value.

## 16. Figures and Tables the Paper Should Include
### Figures
1. Architecture diagram of ROE-TAPE.
2. Reward curriculum and expert activation timeline.
3. Router behavior across market regimes.
4. Training stability curves comparing Run18 vs ROE-TAPE.
5. Regime-wise performance heatmap.
6. Universe comparison chart.

### Tables
1. Main performance comparison table.
2. Architecture ablation table.
3. Router ablation table.
4. Critic ablation table.
5. Covariance feature ablation table.
6. Universe ablation table.
7. Numerical stability / failure incidence table.

### Qualitative diagnostics
- example regime windows with router weights
- example expert preference patterns
- example cases where discipline expert suppresses churn


## 17. What the Paper Should Explicitly Claim
### Strong claim
A staged, regime-routed, multi-objective architecture is a better match for portfolio RL than a monolithic actor-critic when the objectives include return, risk shaping, and trading discipline.

### Defensible supporting claim
Objective-specific critics and actor heads reduce destructive interference during curriculum transitions and improve regime robustness.

### Practical engineering claim
Simplex-stable Dirichlet training requires explicit numerical hardening when concentration can enter low-alpha regions.

### Claim to avoid
Do not claim that the method is universally state of the art across all portfolio RL settings unless the benchmark suite is broad enough.

The paper should claim a **novel framework** with strong evidence, not a universal monopoly on best performance.


## 18. Threats to Validity and How to Address Them
### Threat 1: too many simultaneous changes
Mitigation:
- architecture ablations
- reward ablations
- critic ablations
- router ablations

### Threat 2: universe-specific overfitting
Mitigation:
- balanced vs best-cap vs tech universes

### Threat 3: seed sensitivity
Mitigation:
- multi-seed reporting
- confidence intervals or seed-wise tables

### Threat 4: evaluation cherry-picking
Mitigation:
- fixed deterministic full-test evaluation
- regime-stratified deterministic evaluation
- regime-stratified stochastic confirmation

### Threat 5: numerical fragility hidden by restarts
Mitigation:
- explicitly report non-finite incidents and stabilization measures


## 19. Recommended Research Roadmap
### Phase 1: Prove the core thesis
Use current Run19-style ROE-TAPE against Run18 and simplified ablations.

Goal:
- show objective separation matters
- show routing matters
- show separate critics matter

### Phase 2: Strengthen generality
Run covariance and universe ablations.

Goal:
- show the framework is not tied to one signal configuration or one universe

### Phase 3: Optional extension paper or appendix
Add graph-aware cross-asset reasoning as a second-stage extension.

Important:
- do **not** mix a new GAT module into the first core paper unless the base ROE-TAPE contribution is already proven
- otherwise attribution becomes weak


## 20. Final Recommended Objective Statement
### Final objective
**The primary research objective is to establish ROE-TAPE as a publishable, state-of-the-art multi-objective portfolio RL framework in which TAPE is elevated from a scalar reward design to a routed objective-expert control system with separate critics, staged activation, and simplex-stable policy learning.**

### Minimal publishable package
To make the paper strong, the following are non-negotiable:
- Run18 monolithic baseline
- ROE-TAPE full model
- critic separation ablation
- router ablation
- smooth vs abrupt curriculum ablation
- covariance feature ablation
- universe ablation
- regime-stratified deterministic and stochastic evaluation

That is the smallest experiment set that supports the full claim.


In [ ]:
research_checklist = {
    'baseline_ready': False,
    'roe_tape_ready': True,
    'router_ablation_defined': True,
    'critic_ablation_defined': True,
    'curriculum_ablation_defined': True,
    'covariance_ablation_defined': True,
    'universe_ablation_defined': True,
    'regime_deterministic_eval_defined': True,
    'regime_stochastic_eval_defined': True,
    'multi_seed_plan_defined': True,
}

research_checklist
